# Interactive research: trading the ETH/BTC ratio

A scratchpad over `deribit_pairs_backtest.py`. Load the two perpetual feeds, look
at the spread, ask whether it mean-reverts, and backtest a fade under two honest
execution models: taker (cross and pay) and maker (post a limit, fill only when a
print trades through it).

Run it from the `examples/` directory. It defaults to the committed sample in
`../devtools/data/`; for real research, download a few days and point `DATA_DIR`
at them:

```
python deribit_download.py --days 3 --out-dir .
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import deribit_pairs_backtest as bt

DATA_DIR = "../devtools/data"
BASE, QUOTE = "ETH-PERPETUAL", "BTC-PERPETUAL"
BAR = "30s"

## 1. Load the OHLC spread

The high and low of each bar are what the maker fill model needs.

In [ ]:
times, close, high, low, half = bt.load_ohlc_spread(DATA_DIR, BASE, QUOTE, BAR)
sec = pd.Timedelta(BAR).total_seconds()
bpy = 365 * 24 * 3600 / sec
minutes = len(close) * sec / 60
z, rate = bt.signals(close, z_window=100, regime_window=200)
print(f"{len(close)} bars of {BAR} over {minutes/1440:.1f} days, half-spread {half:.2f} bps")

## 2. Look at it: ratio, z-score, and the OU mean-reversion rate

In [ ]:
fig, ax = plt.subplots(3, 1, figsize=(11, 7), sharex=True)
ax[0].plot(times, np.exp(close)); ax[0].set_ylabel("ETH/BTC ratio")
ax[1].plot(times, z, lw=0.7); ax[1].axhline(0, color="k", lw=0.5); ax[1].set_ylabel("z-score")
ax[2].plot(times, rate, lw=0.7); ax[2].axhline(np.nanmedian(rate), color="r", ls=":")
ax[2].set_ylabel("OU rate"); ax[2].set_xlabel("time")
plt.tight_layout()

## 3. Is there mean-reversion? Return autocorrelation across timeframes

Negative lag-1 autocorrelation of the ratio's returns means it reverts. On a
liquid pair this holds at every timeframe, which is why fading is *gross*
profitable.

In [ ]:
for b in ["1s", "5s", "15s", "60s", "300s"]:
    _, c, _, _, _ = bt.load_ohlc_spread(DATA_DIR, BASE, QUOTE, b)
    ac = pd.Series(np.diff(c)).autocorr(1)
    print(f"{b:>5}: {len(c):>7} bars   lag-1 autocorr {ac:+.3f}")

## 4. Backtest: taker vs maker fill model

Taker crosses at the close and pays both fees plus the spread. Maker posts a
limit at the touch and fills only when a later print trades through it, so it
often misses. Watch the fill rate and the out-of-sample column.

In [ ]:
def table(entry=1.0, exit=0.25, maker_fee=0.0):
    taker_cost = 2 * 5.0 + half
    cut = int(len(close) * 2 / 3)
    print(f"{'strategy':<14} {'gross_S':>8} {'taker%':>8} {'maker%':>8} {'maker_S':>8} {'fill':>5} {'test_mk_S':>9} {'min/trd':>8}")
    for name, gate in [("always-fade", None), ("regime-gated", rate)]:
        gross, _ = bt.taker_backtest(close, z, gate, entry, exit, 0.0, bpy)
        taker, _ = bt.taker_backtest(close, z, gate, entry, exit, taker_cost, bpy)
        maker, fills, fr = bt.maker_backtest(close, high, low, z, gate, entry, exit, half, maker_fee, bpy)
        gate_t = None if gate is None else gate[cut:]
        mk_test, _, _ = bt.maker_backtest(close[cut:], high[cut:], low[cut:], z[cut:], gate_t, entry, exit, half, maker_fee, bpy)
        print(f"{name:<14} {gross['sharpe']:>8.1f} {taker['total']*100:>8.2f} {maker['total']*100:>8.2f} "
              f"{maker['sharpe']:>8.1f} {fr:>5.0%} {mk_test['sharpe']:>9.1f} {minutes/max(fills,1):>8.1f}")

table(entry=1.0, exit=0.25)

In [ ]:
# equity curves for the fill model vs taker
maker_af, _, _ = bt.maker_backtest(close, high, low, z, None, 1.0, 0.25, half, 0.0, bpy)
maker_rg, _, _ = bt.maker_backtest(close, high, low, z, rate, 1.0, 0.25, half, 0.0, bpy)
taker_rg, _ = bt.taker_backtest(close, z, rate, 1.0, 0.25, 2*5.0+half, bpy)
t = times[1:]
plt.figure(figsize=(11, 5))
plt.plot(t, np.cumsum(maker_rg["per_bar"])*100, color="#28a745", label="regime-gated (maker)")
plt.plot(t, np.cumsum(maker_af["per_bar"])*100, color="#0074a2", label="always-fade (maker)")
plt.plot(t, np.cumsum(taker_rg["per_bar"])*100, color="#d62728", ls="--", label="regime-gated (taker)")
plt.axhline(0, color="k", lw=0.5); plt.legend(); plt.grid(alpha=0.25)
plt.ylabel("cumulative P&L (%)"); plt.title("taker vs maker fill model")

## 5. Sensitivity: maker fee

The fill model has no `capture` fudge, so the remaining knobs are the estimated
half-spread and the maker fee. A small maker fee eats the thin edge quickly.

In [ ]:
for fee in [0.0, 0.5, 1.0, 2.0]:
    m, fills, fr = bt.maker_backtest(close, high, low, z, rate, 1.0, 0.25, half, fee, bpy)
    print(f"maker fee {fee:>4.1f} bps/leg   total {m['total']*100:>6.2f}%   Sharpe {m['sharpe']:>6.1f}   fill {fr:.0%}")

## 6. Frequency vs edge: sweep the entry threshold

In [ ]:
print(f"{'entry':>6} {'min/trade':>10} {'maker%':>9} {'fill':>6} {'test_mk_S':>10}")
cut = int(len(close) * 2 / 3)
for entry in [0.5, 0.75, 1.0, 1.5, 2.0]:
    ex = entry * 0.25
    m, fills, fr = bt.maker_backtest(close, high, low, z, None, entry, ex, half, 0.0, bpy)
    mt, _, _ = bt.maker_backtest(close[cut:], high[cut:], low[cut:], z[cut:], None, entry, ex, half, 0.0, bpy)
    print(f"{entry:>6.2f} {minutes/max(fills,1):>10.1f} {m['total']*100:>9.2f} {fr:>6.0%} {mt['sharpe']:>10.1f}")

## Notes and caveats

- **The fill model still flatters you.** It assumes you fill whenever a print
  trades through your level, ignoring queue priority: in reality you also need
  the trades at your price to clear the orders ahead of you. So the ~1/3 fill
  rate here is an optimistic ceiling.
- **Leg risk is folded away.** The spread is treated as one instrument with a
  combined spread; a real maker posts on both legs and can fill one without the
  other.
- **Out of sample is the honest column.** The fade reverts most of the time, but
  when the ratio trends it gives back, and three days is not a sample. Download
  more and re-run.
- Fees are placeholders; set your Deribit tier. Research, not trading advice.